In [121]:
import os
from datetime import datetime
from typing import Annotated, TypedDict, Literal, Optional, List, Dict

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from dotenv import load_dotenv
from loguru import logger

load_dotenv()




True

In [122]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

template = """When asked to ask the user a question, respond with only the question itself, without any additional text or context.

When asked to say goodbye to the user, respond with only a goodbye message.


More instructions:
- Do not add sure to your response.
- Do not add any additional text or context to your response.
- Do not add any additional questions to your response.

For all other interactions:
Your job is to get information from a user about what type of prompt template they want to create.
"""




In [123]:
def get_messages_info(messages):
    return [SystemMessage(content=template)] + messages

# 1. Define a simplified transit state 
class TransitState(TypedDict):
    bool0: bool
    bool1: bool
    bool2: bool
    bool3: bool
    stop_id: str = Optional[str] # for getting stop information

    messages: Annotated[list, add_messages]
    isRunning: Optional[bool] = None
    current_query: Optional[str] = None

class StopInfo(BaseModel):
    ETA: str = None
    Cross_Street: str = None
    Nearest_Highway: str = None
    is_delayed: bool = None
    Delay_Reason: str = None
    location: str = None
    issue: bool



In [124]:
# Helper function
def set_bits(n: int):
    return {
        "bool3": bool(n & 0b1000),
        "bool2": bool(n & 0b0100),
        "bool1": bool(n & 0b0010),
        "bool0": bool(n & 0b0001),
    }

# Nodes
def ask_scheduled_message(state: TransitState):
    print(state)
    print("Current Node: ask_scheduled_message")
    LLM_query = "Are you on track with the delivery?"
    return {**state, **set_bits(0b0001), "current_query": LLM_query}

def get_scheduled_message(state: TransitState):
    print(state)
    print("Current Node: get_scheduled_message")
    human_response = interrupt({"query": state["current_query"]})
    msg = StopInfo(issue=False)
    if not msg.issue:
        return {**state, **set_bits(0b0010)}  # Success -> ask_location
    else:
        return {**state, **set_bits(0b0000)}  # Failure -> ask_scheduled_message

def ask_location(state: TransitState):
    print(state)
    print("Current Node: ask_location")
    LLM_query = "What is your current location?"
    return {**state, **set_bits(0b0011), "current_query": LLM_query}

def get_location(state: TransitState):
    print(state)
    print("Current Node: get_location")
    human_response = interrupt({"query": state["current_query"]})
    if '000' in human_response:
        return {**state, **set_bits(0b0100)}  # Success -> ask_cross_street
    else:
        return {**state, **set_bits(0b0010)}  # Failure -> ask_location

def ask_cross_street(state: TransitState):
    print(state)
    print("Current Node: ask_cross_street")
    LLM_query = "What is your nearest cross street?"
    return {**state, **set_bits(0b0101), "current_query": LLM_query}

def get_cross_street(state: TransitState):
    print(state)
    print("Current Node: get_cross_street")
    human_response = interrupt({"query": state["current_query"]})
    if '000' in human_response:
        return {**state, **set_bits(0b0110)}  # Success -> ask_nearest_highway
    else:
        return {**state, **set_bits(0b0100)}  # Failure -> ask_cross_street

def ask_nearest_highway(state: TransitState):
    print(state)
    print("Current Node: ask_nearest_highway")
    LLM_query = "What is your nearest highway?"
    return {**state, **set_bits(0b0111), "current_query": LLM_query}

def get_nearest_highway(state: TransitState):
    print(state)
    print("Current Node: get_nearest_highway")
    human_response = interrupt({"query": state["current_query"]})
    if '000' in human_response:
        return {**state, **set_bits(0b1000)}  # Success -> ask_eta
    else:
        return {**state, **set_bits(0b0110)}  # Failure -> ask_nearest_highway

def ask_eta(state: TransitState):
    print(state)
    print("Current Node: ask_eta")
    LLM_query = "What is the estimated time of arrival?"
    return {**state, **set_bits(0b1001), "current_query": LLM_query}

def get_eta(state: TransitState):
    print(state)
    print("Current Node: get_eta")
    human_response = interrupt({"query": state["current_query"]})
    if '000' in human_response:
        return {**state, **set_bits(0b1010)}  # Success -> ask_delay
    else:
        return {**state, **set_bits(0b1000)}  # Failure -> ask_eta

def ask_delay(state: TransitState):
    print(state)
    print("Current Node: ask_delay")
    LLM_query = "What is the delay reason?"
    return {**state, **set_bits(0b1011), "current_query": LLM_query}

def get_delay(state: TransitState):
    print(state)
    print("Current Node: get_delay")
    human_response = interrupt({"query": state["current_query"]})
    if '000' in human_response:
        # You could define a next step beyond get_delay
        return {**state, **set_bits(0b1100)}  # No next step specified yet
    else:
        return {**state, **set_bits(0b1010)}  # Failure -> ask_delay


In [125]:
def goodBye(state: TransitState):
    'Say good bye to the user.'
    # msg = llm.invoke('Say good bye to the user.')
    # print(msg.content)
    return {**state, "isRunning": False}

def transit(state: TransitState):
    return state


In [126]:
# Map 4-bit binary codes to actions
state_actions = {
    0b0000: "ask_scheduled_message",
    0b0001: "get_scheduled_message",
    0b0010: "ask_location",
    0b0011: "get_location",
    0b0100: "ask_cross_street",
    0b0101: "get_cross_street",
    0b0110: "ask_nearest_highway",
    0b0111: "get_nearest_highway",
    0b1000: "ask_eta",
    0b1001: "get_eta",
    0b1010: "ask_delay",
    0b1011: "get_delay",
}

def encode_state(state: TransitState) -> int:
    # Encode into a 4-bit integer
    bits = (
        (state["bool3"] << 3) |   # <-- New higher-order bit
        (state["bool2"] << 2) |
        (state["bool1"] << 1) |
        (state["bool0"])
    )
    return bits

def transit_router(state: TransitState) -> str:
    code = encode_state(state)
    return state_actions.get(code, "goodBye")



In [127]:
# 3. Build the graph
def build_transit_graph():
    graph_builder = StateGraph(TransitState)
    
    # Add all the nodes
    graph_builder.add_node("transit", transit)
    graph_builder.add_node("ask_scheduled_message", ask_scheduled_message)
    graph_builder.add_node("get_scheduled_message", get_scheduled_message)
    graph_builder.add_node("ask_location", ask_location)
    graph_builder.add_node("get_location", get_location)
    graph_builder.add_node("ask_cross_street", ask_cross_street)
    graph_builder.add_node("get_cross_street", get_cross_street)
    graph_builder.add_node("ask_nearest_highway", ask_nearest_highway)
    graph_builder.add_node("get_nearest_highway", get_nearest_highway)
    graph_builder.add_node("ask_eta", ask_eta)
    graph_builder.add_node("get_eta", get_eta)
    graph_builder.add_node("ask_delay", ask_delay)
    graph_builder.add_node("get_delay", get_delay)
    graph_builder.add_node("goodBye", goodBye)

    graph_builder.add_edge(START, "transit")

    graph_builder.add_conditional_edges(
        "transit",
        transit_router,
        {
            "ask_scheduled_message": "ask_scheduled_message",
            "get_scheduled_message": "get_scheduled_message",
            "ask_location": "ask_location",
            "get_location": "get_location",
            "ask_eta": "ask_eta",
            "get_eta": "get_eta",
            "ask_delay": "ask_delay",
            "get_delay": "get_delay",
            "ask_cross_street": "ask_cross_street",
            "get_cross_street": "get_cross_street",
            "ask_nearest_highway": "ask_nearest_highway",
            "get_nearest_highway": "get_nearest_highway",
            "goodBye": "goodBye"
        }
    )    

    graph_builder.add_edge("get_delay", "goodBye")
    graph_builder.add_edge("goodBye", END)

    memory = MemorySaver()
    graph = graph_builder.compile(checkpointer=memory)

    return graph




  # Running the Graph

In [128]:
state = {
    "bool3": False,
    "bool2": False,
    "bool1": False,
    "bool0": False,

    "messages": [HumanMessage(content="Hello, how are you?")],
    "current_query": None
}

transit_graph = build_transit_graph()
thread = {"configurable": {"thread_id": "1"}}



  1. Invoke

In [129]:
# result = transit_graph.invoke(state, config = thread)
# result



  2. State Update

In [130]:
# state = transit_graph.get_state(thread).values
# state



  3. Ask + Interrupt

In [131]:
# # ask message
# for event in transit_graph.stream(state, config = thread):
#     print(event)
#     print()



  4. Respond

In [132]:
# # response message
# for event in transit_graph.stream(
#     Command(resume="the driver's late"), 
#     config = thread
# ):
#     print(event)
#     print()



  5. State Update

In [133]:
# state = transit_graph.get_state(thread).values
# state



  Repeat

In [134]:
# result = transit_graph.invoke(state, config = thread)
# result



In [135]:
# state = transit_graph.get_state(thread).values
# state



In [136]:
# # ask message
# for event in transit_graph.stream(state, config = thread):
#     print(event)
#     print()



In [137]:
# # response message
# for event in transit_graph.stream(
#     Command(resume="yes"), 
#     config = thread
# ):
#     print(event)
#     print()



  Combine

In [138]:
# result = transit_graph.invoke(state, config = thread)
# result

# state = transit_graph.get_state(thread).values
# state

# # ask message
# for event in transit_graph.stream(state, config = thread):
#     print(event)
#     print()

# # response message
# for event in transit_graph.stream(
#     Command(resume="yes"), 
#     config = thread
# ):
#     print(event)
#     print()

# state = transit_graph.get_state(thread).values
# state



In [139]:
from IPython.display import Image, display

try:
    display(Image(transit_graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass



  # Utility functions

In [140]:
def format_response(response, state: TransitState):
    """Format the response for the UI"""
    message = ""
    
    # Log the exact response for debugging
    logger.debug(f"Raw response type: {type(response)}")
    logger.debug(f"Raw response: {response}")
    
    try:
        # Handle dictionary responses
        if isinstance(response, dict):
            # Handle interrupt messages
            if '__interrupt__' in response:
                interrupt_tuple = response['__interrupt__']
                if isinstance(interrupt_tuple, tuple) and len(interrupt_tuple) > 0:
                    interrupt_obj = interrupt_tuple[0]
                    if hasattr(interrupt_obj, 'value') and isinstance(interrupt_obj.value, dict):
                        message = interrupt_obj.value.get('query', '')
                        logger.debug(f"Extracted query from interrupt: {message}")
            # Handle ask_delay responses
            elif 'ask_delay' in response:
                delay_info = response['ask_delay']
                if isinstance(delay_info, dict) and 'current_query' in delay_info:
                    message = delay_info['current_query']
                    logger.debug(f"Extracted query from ask_delay: {message}")
            # Handle goodbye messages
            elif 'goodBye' in response:
                message = "Goodbye! Thank you for using our service."
                logger.debug("Generated goodbye message")
        
        # Handle string responses (fallback)
        elif isinstance(response, str):
            message = response
            logger.debug(f"Using string response directly: {message}")

    except Exception as e:
        logger.error(f"Error formatting response: {e}")
        message = str(response)

    # If we still don't have a valid message, use a safe fallback
    if not message:
        logger.warning(f"Failed to extract message, using raw response: {response}")
        message = str(response)

    logger.debug(f"Final formatted message: {message}")
    
    # Ensure all required state fields are present
    state_dict = {
        "bool2": state.get("bool2", False),
        "bool1": state.get("bool1", False),
        "bool0": state.get("bool0", False)
    }
    
    # Determine current step
    if not state_dict["bool2"]:
        current_step = "scheduled"
    elif not state_dict["bool1"]:
        current_step = "location"
    elif not state_dict["bool0"]:
        current_step = "eta"
    else:
        current_step = "complete"
    
    return {
        "message": message,
        "state": {
            **state_dict,
            "current_step": current_step
        }
    }
def process_transit_chat_sequence(state: TransitState, user_message: Optional[str] = None):
    """
    Process the transit chat sequence following the pattern:
    1. interrupt - get question for user
    2. update state - handle user's response
    3. simple query - process current state
    4. new interrupt - prepare next question
    5. repeat
    """
    thread = {"configurable": {"thread_id": "1"}}
    
    # Ensure all required fields are present in state
    if not isinstance(state, dict):
        state = initialize_transit_chat()
    else:
        # Add any missing fields with default values
        default_state = initialize_transit_chat()
        for key in default_state:
            if key not in state:
                state[key] = default_state[key]
    
    if user_message is None:
        # Initial flow - get first interrupt
        result = transit_graph.invoke(state, config=thread)
        response = None
        for event in transit_graph.stream(result, config=thread):
            response = event
        return format_response(response, state)
    else:
        # 1. Handle interrupt (user's response)
        result = transit_graph.invoke(Command(resume=user_message), config=thread)
        
        # 2. Update state
        state = transit_graph.get_state(thread).values
        
        # 3. Process current state
        response = None
        for event in transit_graph.stream(state, config=thread):
            if event:
                response = event
                
        # 4. Get new interrupt/next state
        if not state["bool2"] or not state["bool1"] or not state["bool0"]:
            result = transit_graph.invoke(state, config=thread)
            for event in transit_graph.stream(result, config=thread):
                if event:
                    response = event
        
        return format_response(response, state)

def initialize_transit_chat():
    """Initialize a new transit chat session with default state"""
    initial_state = {
        "bool3": False,
        "bool2": False,
        "bool1": False,
        "bool0": False,
        
        "messages": [HumanMessage(content="Hello, how are you?")],
        "current_query": None
    }
    return initial_state

def get_current_state():
    """Get the current state of the conversation"""
    thread = {"configurable": {"thread_id": "1"}}
    return transit_graph.get_state(thread).values

# Example usage in the notebook:
"""
# Initialize chat
state = initialize_chat()

# Get first interrupt/question
response = process_chat_sequence(state)
print("Bot:", response)

# Send user response and get next interrupt
response = process_chat_sequence(state, "yes")
print("Bot:", response)

# Continue conversation...
response = process_chat_sequence(state, "yes")
print("Bot:", response)
"""











'\n# Initialize chat\nstate = initialize_chat()\n\n# Get first interrupt/question\nresponse = process_chat_sequence(state)\nprint("Bot:", response)\n\n# Send user response and get next interrupt\nresponse = process_chat_sequence(state, "yes")\nprint("Bot:", response)\n\n# Continue conversation...\nresponse = process_chat_sequence(state, "yes")\nprint("Bot:", response)\n'